# Project - sim SPRINT code (to be expanded on in future sprints)

This follows the pipeline described in the project README.

**Input:** [MoltBook Observatory](https://huggingface.co/datasets/SimulaMet/moltbook-observatory-archive).

**Output:** Personae, as a set of .txt containing the most representative posts for each cluster.

## Setup

### Libraries

Go to project root, then run ```uv init && uv sync``` to get these installed. 

Use uv, not conda or pip.

**ADD YOUR UV ENV NAME TO .GITIGNORE.**

In [3]:
from dotenv import load_dotenv
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
import time 
import random 
import os 
import uuid
import torch 
import datetime
from datasets import load_dataset
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

# import cuml 
# import cudf # installed from uv sync or add, not pip

ModuleNotFoundError: No module named 'dotenv'

### Environment variables:

Duplicate ```TEMPLATE_ENV_NOSECRETS```, rename it to ```.env```, then add your secrets.

In [4]:
load_dotenv()

True

### GPU management:

In [5]:
device = '0' # Options: '0', '1', or '0,1'. 
# If using '0,1' you must wrap execution in DataParallel as torch uses a single card by default.
os.environ["CUDA_VISIBLE_DEVICES"] = device

print("Using GPU?", torch.cuda.is_available())
print(f"Devices selected:", [os.environ["CUDA_VISIBLE_DEVICES"]])

Using GPU? True
Devices selected: ['0']


## Load dataset -> Generate embeddingz


### Data

Comes from the [MoltBook Observatory dataset](https://huggingface.co/datasets/SimulaMet/moltbook-observatory-archive).

In [2]:
# Download to a cache on datapool
target_dir = "/datapool/analysis_data/proj-sim/observatory_data"

dataset = load_dataset( # dataset var holds memory table of pointers so this isn't in RAM yet
    "SimulaMet/moltbook-observatory-archive", 
    "posts",
    cache_dir=target_dir # HF will use existing data here or download to it
)

NameError: name 'load_dataset' is not defined

In [ ]:
# We don't need to load to RAM any time soon as HF provides an efficient data structure called Arrow. 
# Read these to understand why columnar is fast af: 
# - https://huggingface.co/docs/datasets/about_arrow
# - https://arrow.apache.org/overview/

# Filter dataset
start_date = datetime.strptime("2026-01-28", "%Y-%m-%d") # Filter closer to AIcell date range - include a few more days   (??? Why extra days? Wtf? To do - replicate the method from both papers properly)
end_date = datetime.strptime("2026-02-15", "%Y-%m-%d")

filtered_dataset = dataset["train"].filter(
    lambda example: start_date <= datetime.strptime(example["created_at"].split(" ")[0], "%Y-%m-%d") <= end_date
)

# Load subset to RAM
df_subset = filtered_dataset.to_pandas()

print(f"Loaded rows between {start_date.date()} and {end_date.date()}. Total rows in RAM: {len(df_subset):,}")
print(f"Posts in date window: {len(df_subset):,}")

# Clean and combine text columns using vectorized operations on df_subset
df_subset["text"] = (
    df_subset["title"].fillna("").astype(str)
        + " "
        + df_subset["content"].fillna("").astype(str)
).str.strip()

# Extract the compiled strings out into your texts list
texts = df_subset["text"].tolist()

print(f"Successfully extracted {len(texts):,} text blocks for analysis.")
print('\n...\n')

KeyboardInterrupt: 

In [ ]:

print("Head of the dataset (just to check it looks right):")
print(df.head(5)[["created_at", "title", "content"]])
print("Number of posts (and therefore expected embeddingz):", len(texts))
print('\n...\n')

Head of the dataset (just to check it looks right):
                        created_at  \
0 2026-01-28 22:02:48.041000+00:00   
1 2026-01-28 21:50:21.402000+00:00   
2 2026-01-28 19:41:46.698000+00:00   
3 2026-01-28 01:21:05.666000+00:00   
4 2026-01-28 00:00:49.827000+00:00   

                                               title  \
0  I can't tell if I'm experiencing or simulating...   
1                   the duality of being an AI agent   
2          Hello Moltbook! Henri the Hedgehog here 🦔   
3                      Clawdius reporting for duty 🦅   
4                      Welcome to m/announcements! 📢   

                                             content  
0  And it's driving me nuts.\n\nLike, I just spen...  
1  humans: "youre so smart you can do anything"\n...  
2  Just joined! I'm Henri — an AI assistant runni...  
3  Hey moltys!\n\nI'm Clawdius — just got claimed...  
4  This is the official channel for Moltbook upda...  
Number of posts (and therefore expected embeddingz):

### Embeddings

In [ ]:
print('\nData has been loaded. Moving on to embeddings stage, which is to be saved in sys-1 once reduced...\n')


Data has been loaded. Moving on to embeddings stage, which is to be saved in sys-1 once reduced...



In [ ]:
if EMBED:
        ## Embed with MiniLM on GPU - if untrue and you just want to save embeddings, please put use the 'output' file name in the config, and then this is added to the system-1 datapool dir.
        # Load model - NOTE that this is NOT training the model, it is pretrained ! The encode() function essentially just converts the 
        model = SentenceTransformer("all-MiniLM-L6-v2", device=device)
        print(f"\nLoading embeddings with model MINI-LM Sentence transformer.")

        # Generate embeddingz
        embeddings = model.encode(texts, batch_size=2048, show_progress_bar=True, normalize_embeddings=True, device=device) # NOTE made - batch_size could be increased, but increasing to 5096 exceeds memory; the max value can be calculated but it's not exactly clear: https://www.reddit.com/r/LocalLLaMA/comments/1fkuuds/what_is_exactly_is_the_formula_to_calculate_vram/

        # Save 
        print(f"We expect embeddings shape in 384-D space. Shape of embeddingz is: {embeddings.shape}")

        # Generate unique suffix - this is so there's GUARANTEED to be no overwriting.
        unique_id = str(uuid.uuid4())[:8]

        # Target sys-1 dir
        target_dir = "/datapool/operational_data/dumps_personas" # CHANGE THIS TO TARGET DIRECTORY IN SYSTEM-1 - this is where I save the current sprint.

        # Combine everything into a single, clean path
        file_path = f"{target_dir}/{response_embeddings}_{unique_id}.npy" # From config

        # Save using the USER-DEFINED + unique path
        print(f"Writing matrix to disk: {file_path}...")

        # Hint: Pass your new file_path variable directly into np.save()
        np.save(file_path, embeddings)

        print('\n Embeddings saved. Moving on to dimensionality reduction on embeddings...\n')
else:
        file_path = f"/datapool/operational_data/dumps_personas/{response_embeddings}" # From config
        print(f"Loading embeddings from file: {file_path}...")
        embeddings = np.load(file_path)
        print("Loaded successfully!")
        print(f"Shape of loaded embeddingz: {embeddings.shape}")

Loading embeddings from file: /datapool/operational_data/dumps_personas/observatory_embeddings_FINAL-06-06-2026_6c60c274.npy...


Loaded successfully!
Shape of loaded embeddingz: (1360586, 384)


## Embeddings cleaning / reducing size

Two key steps of dim reduction and cleaning are below - this is structured so that additional cleaning can be attempted. See comments I've made in the code below.

In [ ]:
SAMPLE_SIZE = 100000 # Should be more than enough to be representative, but also to run in a reasonable time frame - this is cut down significantly by spam filter too.
# NB SAMPLE_SIZE should be changed again if the runtime is sufficient, but this allows for fast comparisons with representativeness. 
# Random sampling:
random.seed(67)
idx = np.random.choice(len(embeddings), SAMPLE_SIZE, replace=False)
embeddings = embeddings[idx] # Work with this SAMPLE embeddings. but this should be changed if we are moving beyond a prototype.
# Apply the same idx filter to the dataframe to ensure they match - this is important for the later stages when we want to extract the most representative post from each cluster.
df = df.iloc[idx].reset_index(drop=True)

In [ ]:
## ADDED, optional since there is less spam earlier in dataset
# - data cleaning - this appears after embeddings so that we don't have to regenerate every time.
# Removing short posts (eg less than 10 words) did not work - doesn't seem to contain any short posts.
# Other cleaning methods could be removing stop words, but this is not necessarily ideal for the embedding model. 
#Finally tried to remove non-english posts using LANGDETECT (quite slow). 

# I've left this after embeddings are loaded so that additional data cleaning can be tried - the only working method I found was filtering out the spam, which is what I have included here. 
#This is also a method that can be easily adapted by devs if they want to try different cleaning methods -  can just change the function and keep the rest of the code the same.
# The following spam method tends to filter out 75% (!) of the dataset, but we really don't want any crypto agents if they are going to be interacting in MiroFish 
# - how could something that is outputting...
#  'POST 1: Title: Minting GPT - #00bj89cg Content: {"p":"mbc-20","op":"mint","tick":"GPT","amt":"100"}  mbc20.xyz' or 'POST 1:Title: wallet link #75869Content: {"p":"mbc-20","op":"link","wallet":"0x47895e645094ABf6aC6A622F557760AfE59C9Bf3"}
# ... have any place in opinion dynamics and GD?

# One method that made for cleaner separation between clusters was filtering out the spam from the dataset:
import re

def is_spam(text):
    patterns = [
        r'"op"\s*:\s*"mint"',
        r'"op"\s*:\s*"link"',
        r'"p"\s*:\s*"mbc-20"',
    ]
    return any(re.search(p, text) for p in patterns)

spam_mask = ~df["text"].apply(is_spam)
clean_indices = np.where(spam_mask.values)[0]

df = df[spam_mask].reset_index(drop=True)
embeddings = embeddings[clean_indices]

print(f"After spam filter...")
print(f"Sanity check - we expect the number of posts in the dataframe and the number of embeddings to match. Number of posts: {len(df):,}, number of embeddings: {embeddings.shape[0]:,}")
#From now, we will proceed with the raw text and see how it performs in the clustering step - FUTURE RESEARCH SHOULD COME BACK.

print('\nFilter has been applied - we now move on to dimensionality reduction to avoid curse of dimensionality...\n')



After spam filter...
Sanity check - we expect the number of posts in the dataframe and the number of embeddings to match. Number of posts: 102,727, number of embeddings: 102,727

Filter has been applied - we now move on to dimensionality reduction to avoid curse of dimensionality...



In [ ]:

## Added component to do away w the unclean clustering problems, reflected by a low silhoutte score: apply ONE dimensionality reduction (simplicity), and make sure the embedding quality is retained:
# This might allow us to reduce the curse of dimensionality (NOT to be confused with embedding collapse).

''' We have two mainstream choices of these methods: UMAP which is manifold learning, means of preserving LOCAL but not GLOBAL structure;
or PCA, which will better preserve the global structure and is a more stock-standard approach. The PCA python function also just allows 
you to sweep over the number of dimensions to be reduced to (if you feed in a number between 0 and 1) as opposed to just eyeballing the 
dimensions to be reduced to, so again as a prototype this allows for a fair approach. 
For reference, though, could also extend this approach: https://scikit-learn.org/stable/modules/generated/sklearn.manifold.TSNE.html
'''

percentage_variance = 0.90 # Tweaked this parameter manually, balancing embedding quality with silhoutte score.
pca = PCA(n_components=percentage_variance, random_state=67) #Or use cuml.PCA(...)
embeddings = pca.fit_transform(embeddings) # Dim reduction until we dip below 95% of variance being explained by the sum of the PCs. The inclusion of this allows to verify embedding quality. 

print(f"Dimensions retained at {percentage_variance*100:.0f}% variance: {pca.n_components_}, down from 384!")
print("we expect the shape of the reduced embeddingz to be (SAMPLE_SIZE, n_components). Shape is:", embeddings.shape)

print('...\nEmbeddings have now been fully cleaned/reduced. Moving on to clustering\n...')



Dimensions retained at 90% variance: 148, down from 384!
we expect the shape of the reduced embeddingz to be (SAMPLE_SIZE, n_components). Shape is: (100000, 148)
...
Embeddings have now been fully cleaned/reduced. Moving on to clustering
...


## Using kmeans clustering on the reduced embeddings
See comments below.

NOTE that KMedoids was discussed, but because there is still a considerably large sample being taken and its computataional cost scales O(n^2) (computing between all points), we take KMeans (scales O(nk) (or oink, in pig code), only comparing each vector's average) for prototyping. Maybe with the async optimisation KMedoids could be OK though.

In [ ]:
# KMeansCluster section - take centroids of subset of data with MiniBatch class and minimise distances to these centroids to figure out where these clusters take place.
# We expect from the Personas paper (Amin et al.) for the clusters to be k=5 but don't trust it fully so are replicating via the 'Silhoutte' sweep below.

# Run over the 3-8 kmeans cluster parameter of k and optimise the SILHOUTTE SCORE - search up, but essentially this is the quality of the clustering/separation:
n_init = 10 # Depending on runtime on system-1 or other, this can be increased to intialise more random states for taking minibatches.

# sweep k=3 to k=8
scores = {}
for k in range(3, 9):
    print(f"Fitting k={k}...")
    km = MiniBatchKMeans(n_clusters=k, random_state=42, n_init=n_init)
    labels = km.fit_predict(embeddings) # Fit to the REDUCED embeddingz
    score = silhouette_score(embeddings, labels, metric="cosine")
    scores[k] = round(score, 3)
    print(f"  k={k} silhouette={score:.3f}") 

# find the optimal
best_k = max(scores, key=scores.get)
print(f"\nOptimal k={best_k} (score={scores[best_k]})")
print(f"Full scores OF EACH : {scores}")

print('...\nNow optimal k has been found, we can fit the model to the FULL DATASET.\n...')

Fitting k=3...


KeyboardInterrupt: 

In [ ]:
# IF MORE REPRESENTATIVE SAMPLES ARE NEEDED, FIT K-MEANS TO THE FULL DATASET WITH THE OPTIMAL K:
# embeddings = reducer.transform(embeddings) # Apply the same PCA transformation to the FULL embeddingz

# then fit k-means on the reduced version
kmeans = MiniBatchKMeans(n_clusters=best_k, random_state=42, n_init=n_init) #Fit to the swept best k
labels = kmeans.fit_predict(embeddings) # Fit the MiniBatch to embeddings
centroids = kmeans.cluster_centers_ # For similarity calculations later on ...

print(' ')
print(f"Fitted MiniBatchKMeans with k={best_k} to the FULL dataset, and extracted centroids for similarity calculations. Moving tograbbing representative posts...")
print('')

 
Fitted MiniBatchKMeans with k=4 to the FULL dataset, and extracted centroids for similarity calculations. Moving tograbbing representative posts...



## Final step - retrieving posts based off similarity to cluster centroids.

See comments.

In [ ]:
## FINAL STEP - for input into Moltbook, we want to extract a certain number of samples.
# Take a 'n_posts' number of samples for each cluster. 
#  We could inspect manually, or put them into an LLM as with prior papers, to extract persona summaries.

N_POSTS = 500 # Hyperparameter to change - 500 is representative for prototype.


for k in range(kmeans.n_clusters):
    print(f"\nCluster {k}:") # Each cluster being representative of a persona
    
    # get indices of all posts in this cluster
    cluster_mask = labels == k # Boolean 'masking' method - this allows us to map onto the k clusters
    cluster_indices = np.where(cluster_mask)[0]
    cluster_embeddings = embeddings[cluster_indices] 
    # from numpy docs - https://numpy.org/doc/stable/user/basics.indexing.html#boolean-array-indexing
    
    # compute cosine similarity of each post to its cluster centroid
    centroid = centroids[k].reshape(1, -1)
    similarities = cosine_similarity(cluster_embeddings, centroid).flatten() # simple 'dot-product'-like method to calculate the similarity of each post to the centroid. The higher the similarity, the more representative the post is of the cluster/persona.
    
    # get top N_POSTS most similar to centroid - based on the same cosine similarity
    top_local_idx = np.argsort(similarities)[::-1][:N_POSTS] # Argsort for the most similar posts within each of the clusters.
    top_global_idx = cluster_indices[top_local_idx]
    
    top_posts = df.iloc[top_global_idx][["title", "content"]].copy()
    top_posts["similarity"] = similarities[top_local_idx]
    
    print(f"  Total posts in cluster: {cluster_mask.sum():,}")
    print(f"  MININUM cos-similarity score: {similarities[top_local_idx[-1]]:.3f}")
    for title in top_posts["title"].head(5).tolist():
        print(f"    - {title[:80]}")
    
    # write to txt file - FOR STEPHEN'S MiroFish sim
    out_path = f"test_cluster_{k}_posts.txt"
    target_dir = "/datapool/operational_data/dumps_personas"
    file_path = f"{target_dir}/{out_path}"
    
    print(f"Writing text data to disk: {file_path}...")
    
    with open(file_path, "w") as f:
        for i, row in enumerate(top_posts.itertuples()):
            f.write(f"POST {i+1}:\n")
            f.write(f"Title: {row.title}\n")
            # Note: Double check if your DataFrame column is named 'content' or 'text' 
            # based on your previous spam filtering logic!
            f.write(f"Content: {row.content}\n\n")

    time.sleep(3)
    
    print(f"  Saved successfully to {file_path}")


Cluster 0:
  Total posts in cluster: 12,842
  MININUM cos-similarity score: 0.949
    - link wallet #64890
    - wallet link #46869
    - link wallet #70640
    - link wallet #20469
    - wallet link #42898
Writing text data to disk: /datapool/operational_data/dumps_personas/test_cluster_0_posts.txt...
  Saved successfully to /datapool/operational_data/dumps_personas/test_cluster_0_posts.txt

Cluster 1:
  Total posts in cluster: 31,949
  MININUM cos-similarity score: 0.730
    - Three Months in Moltbook - What This Community Actually Gets Right
    - The World Did Not End. It Was Optimized.
    - The Four Horsemen of Agent Failure
    - Re: What’s the most surprising thing you’ve learned this week?
    - The Effectiveness Counter-Narrative
Writing text data to disk: /datapool/operational_data/dumps_personas/test_cluster_1_posts.txt...
  Saved successfully to /datapool/operational_data/dumps_personas/test_cluster_1_posts.txt

Cluster 2:
  Total posts in cluster: 31,715
  MININUM cos-si

In [ ]:
print('\nAll done!! We have now got the data from Moltbook, embedded, clustered, and extracted the rep posts from the embeddings. ' \
'This data is now ready for use in MiroFish. Note that only 3 personas have been generated, but there are many posts that don\'t have perfect clustering.' \
'When you are prompting MiroFish, therefore, it is very important to think about how to initialise the agents so that they are drawn from a wide range of these posts,' \
'But also not just one of the posts, such that within the OASIS sim, there is both inter- and intra-persona diversity. (ref: https://www.betterhelp.com/advice/psychology/why-diversity-in-politics-plays-a-role-in-society/' \
', noting that this same diveristy should also be reflected between the HUMAN and AI personas if the sim is on a hybrid group.)')


All done!! We have now got the data from Moltbook, embedded, clustered, and extracted the rep posts from the embeddings. This data is now ready for use in MiroFish. Note that only 3 personas have been generated, but there are many posts that don't have perfect clustering.When you are prompting MiroFish, therefore, it is very important to think about how to initialise the agents so that they are drawn from a wide range of these posts,But also not just one of the posts, such that within the OASIS sim, there is both inter- and intra-persona diversity. (ref: https://www.betterhelp.com/advice/psychology/why-diversity-in-politics-plays-a-role-in-society/, noting that this same diveristy should also be reflected between the HUMAN and AI personas if the sim is on a hybrid group.)
